# Entrenamiento de Modelo ASTM - Oxidación y Ampollamiento

Este notebook entrena un modelo de clasificación de imágenes para evaluar:
- **ASTM D610**: Grado de oxidación en superficies pintadas (Grados 0-10)
- **ASTM D714**: Grado de ampollamiento en pinturas (Tamaños 10,8,6,4,2)

## Flujo de Trabajo
1. Montar Google Drive y cargar dataset
2. Preprocesar imágenes con data augmentation
3. Transfer Learning con MobileNetV2
4. Exportar modelo a TensorFlow Lite (.tflite)
5. Guardar en Google Drive para que la app lo descargue

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configurar rutas
DATASET_DIR = "/content/drive/MyDrive/ASTM_Dataset/To_Train"
MODEL_OUTPUT_DIR = "/content/drive/MyDrive/ASTM_Dataset/Models"

import os
print(f"Dataset directory: {DATASET_DIR}")
print(f"Model output directory: {MODEL_OUTPUT_DIR}")

# Crear directorio de salida si no existe
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

In [ ]:
# Instalar dependencias
!pip install tensorflow==2.14.0
!pip install tensorflow-hub==0.14.0
!pip install opencv-python-headless
!pip install scikit-learn

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from datetime import datetime

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Configuración del modelo
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.001
NUM_CLASSES = None  # Se calculará automáticamente

# Verificar estructura de carpetas
def check_dataset_structure(base_dir):
    classes = []
    total_images = 0
    
    if not os.path.exists(base_dir):
        print(f"ERROR: Directorio {base_dir} no existe")
        return [], 0
    
    for class_name in os.listdir(base_dir):
        class_path = os.path.join(base_dir, class_name)
        if os.path.isdir(class_path):
            images = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
            if len(images) > 0:
                classes.append(class_name)
                total_images += len(images)
                print(f"  {class_name}: {len(images)} imágenes")
    
    return classes, total_images

print("\nVerificando estructura del dataset...")
classes, total_images = check_dataset_structure(DATASET_DIR)
NUM_CLASSES = len(classes)
print(f"\nTotal de clases: {NUM_CLASSES}")
print(f"Total de imágenes: {total_images}")

if NUM_CLASSES == 0:
    raise Exception("No se encontraron imágenes en el dataset. Sube fotos corregidas desde la app primero.")

In [ ]:
# Data Augmentation y Preprocesamiento
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomBrightness(0.1),
], name="data_augmentation")

def create_datasets(base_dir, classes, img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    """Crea datasets de entrenamiento y validación"""
    
    image_paths = []
    labels = []
    
    for i, class_name in enumerate(classes):
        class_path = os.path.join(base_dir, class_name)
        if os.path.isdir(class_path):
            for img_file in os.listdir(class_path):
                if img_file.endswith(('.jpg', '.jpeg', '.png')):
                    image_paths.append(os.path.join(class_path, img_file))
                    labels.append(i)
    
    # Dividir en train/validation (80/20)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        image_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    print(f"Imágenes de entrenamiento: {len(train_paths)}")
    print(f"Imágenes de validación: {len(val_paths)}")
    
    def load_and_preprocess_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    
    train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    train_dataset = train_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    val_dataset = val_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    return train_dataset, val_dataset

train_ds, val_ds = create_datasets(DATASET_DIR, classes)

In [ ]:
# Crear modelo con Transfer Learning (MobileNetV2)
def create_model(num_classes, img_size=IMG_SIZE):
    # Base pre-entrenada (MobileNetV2)
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Congelar capas base
    base_model.trainable = False
    
    # Capas personalizadas
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(img_size, img_size, 3)),
        data_augmentation,
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

model, base_model = create_model(NUM_CLASSES)

# Compilar modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model.summary()

In [ ]:
# Callbacks
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

In [ ]:
# Entrenamiento inicial (capas superiores congeladas)
print("\n=== Fase 1: Entrenando capas superiores ===")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_callback, early_stopping, reduce_lr]
)

# Guardar histórico de entrenamiento
import pickle
with open('/content/training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

In [ ]:
# Fine-tuning (descongelar algunas capas)
print("\n=== Fase 2: Fine-tuning ===")

# Descongelar últimas 50 capas
for layer in base_model.layers[:-50]:
    layer.trainable = False
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Recompile con learning rate más bajo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenar por 10 epochs más
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_callback, early_stopping]
)

In [ ]:
# Visualizar resultados
import matplotlib.pyplot as plt

def plot_training_history(history):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Train Acc')
    axes[0, 0].plot(history.history['val_accuracy'], label='Val Acc')
    axes[0, 0].set_title('Accuracy')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    
    # Loss
    axes[0, 1].plot(history.history['loss'], label='Train Loss')
    axes[0, 1].plot(history.history['val_loss'], label='Val Loss')
    axes[0, 1].set_title('Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    
    # Precision
    if 'precision' in history.history:
        axes[1, 0].plot(history.history['precision'], label='Train Prec')
        axes[1, 0].plot(history.history['val_precision'], label='Val Prec')
        axes[1, 0].set_title('Precision')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Precision')
        axes[1, 0].legend()
    
    # Recall
    if 'recall' in history.history:
        axes[1, 1].plot(history.history['recall'], label='Train Rec')
        axes[1, 1].plot(history.history['val_recall'], label='Val Rec')
        axes[1, 1].set_title('Recall')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Recall')
        axes[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig('/content/training_history.png', dpi=300)
    plt.show()

plot_training_history(history)

In [ ]:
# Evaluar modelo en conjunto de validación
print("\n=== Evaluación Final ===")
val_loss, val_acc, val_prec, val_rec = model.evaluate(val_ds, verbose=1)
print(f"\nValidación Accuracy: {val_acc:.4f}")
print(f"Validación Loss: {val_loss:.4f}")
print(f"Validación Precision: {val_prec:.4f}")
print(f"Validación Recall: {val_rec:.4f}")

In [ ]:
# Convertir a TensorFlow Lite
print("\n=== Convirtiendo a TensorFlow Lite ===")

# Cargar mejor modelo
best_model = tf.keras.models.load_model('/content/best_model.h5')

# Convertir
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]  # Cuantización FP16 para menor tamaño

tflite_model = converter.convert()

# Guardar modelo TFLite
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
version_num = len([f for f in os.listdir(MODEL_OUTPUT_DIR) if f.endswith('.tflite')]) + 1
model_name = f"astm_model_v{version_num}_{timestamp}.tflite"
model_path = os.path.join(MODEL_OUTPUT_DIR, model_name)

with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f"\nModelo guardado en: {model_path}")
print(f"Tamaño del modelo: {len(tflite_model) / (1024*1024):.2f} MB")

# Guardar archivo de etiquetas (classes.txt)
classes_path = os.path.join(MODEL_OUTPUT_DIR, f"classes_v{version_num}.txt")
with open(classes_path, 'w') as f:
    for class_name in classes:
        f.write(class_name + '\n')

print(f"Etiquetas guardadas en: {classes_path}")

# También guardar como "latest.tflite" para descarga automática
latest_path = os.path.join(MODEL_OUTPUT_DIR, "latest.tflite")
with open(latest_path, 'wb') as f:
    f.write(tflite_model)
print(f"Copia latest guardada en: {latest_path}")

In [ ]:
# Probar modelo TFLite
print("\n=== Probando modelo TFLite ===")

import numpy as np

# Cargar intérprete TFLite
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Probar con algunas imágenes de validación
test_images = []
test_labels = []

for img_path, label in zip(val_paths[:5], val_labels[:5]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    test_images.append(img)
    test_labels.append(label)

correct = 0
for i, (img, true_label) in enumerate(zip(test_images, test_labels)):
    input_data = np.expand_dims(img, axis=0)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    output = interpreter.get_tensor(output_details[0]['index'])
    predicted_label = np.argmax(output[0])
    confidence = output[0][predicted_label]
    
    is_correct = predicted_label == true_label
    if is_correct:
        correct += 1
    
    print(f"Imagen {i+1}: Predicho={classes[predicted_label]}, Real={classes[true_label]}, Conf={confidence:.2f}, {'✓' if is_correct else '✗'}")

print(f"\nPrecisión en prueba: {correct}/{len(test_images)} = {correct/len(test_images):.2%}")

In [ ]:
# Resumen final
print("\n" + "="*60)
print("ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
print("="*60)
print(f"\n📊 Estadísticas:")
print(f"   - Clases: {NUM_CLASSES}")
print(f"   - Imágenes totales: {total_images}")
print(f"   - Accuracy validación: {val_acc:.2%}")
print(f"\n📦 Archivos generados:")
print(f"   - Modelo TFLite: {model_path}")
print(f"   - Etiquetas: {classes_path}")
print(f"   - Latest model: {latest_path}")
print(f"\n📱 Próximos pasos:")
print(f"   1. La app Android detectará automáticamente el nuevo modelo")
print(f"   2. Al iniciar, descargará 'latest.tflite' de Google Drive")
print(f"   3. El contador de reentrenamiento se reseteará")
print(f"\n✅ ¡Listo para usar!"